# Notebook 03 — Análisis de Componentes Principales (PCA) y Selección de Bandas (Objetivo Específico 1)

- **Instancia SEPAL recomendada:** `t2` o `m2` (0.04–0.10 USD/h).
- **Responde al Objetivo Específico 1:** *"Identificar las bandas espectrales y polarizaciones SAR más efectivas para la correcta clasificación del bosque manglar."*
- **Qué produce este notebook:**
  1. Extracción exacta a 10 m de las **24 bandas del Stack Biestacional** en todos los puntos de control (train + test preliminar) y respaldo en CSV.
  2. **PCA Biestacional (24 bandas)** + **PCA Comparativo por Temporada (12 Seca vs. 12 Lluvia)** sobre puntos de Mangle (estandarización Z-Score):
     - Matriz de Covarianza, Matriz de Correlación de Pearson, Autovalores (Criterio de Kaiser $\lambda > 1$), Scree Plot, Eigenvectores/Loadings y Ranking de Contribución Relativa por Banda.
  3. **Análisis complementario de Separabilidad Multiclase** (firmas espectrales/SAR de las 5 coberturas + ANOVA F-score + distancia de separabilidad *Mangle vs. Bosque latifoliado*) para defender sólidamente el método ante la terna.
  4. Archivo `bandas_seleccionadas_pca_<region>.json` consumido automáticamente por el Notebook 04.

In [ ]:
# 1. Configuracion y carga de insumos
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from src.config_utils import (
    ensure_output_dirs,
    get_class_maps,
    get_predictor_band_names,
    init_gee,
    load_config,
)
from src.plotting import (
    plot_band_contributions,
    plot_correlation_matrix,
    plot_scree_and_variance,
    set_thesis_style,
)
from src.stack_builder import get_stack_asset_ids

# >>> CAMBIAR AQUI ENTRE "PACIFICO" Y "CARIBE" <<<
REGION = "PACIFICO"

cfg = load_config()
dirs = ensure_output_dirs(cfg)
init_gee(cfg)
set_thesis_style()

CLASES_NOMBRES, CLASES_CORTOS, CLASES_COLORES = get_class_maps(cfg)
BANDS = get_predictor_band_names(cfg)
stack_ids = get_stack_asset_ids(REGION, cfg)

print("Stack a muestrear:", stack_ids["stack"])
print(f"Total bandas predictoras: {len(BANDS['todas'])}")

In [ ]:
# 2. Extraer las 24 bandas del Stack Biestacional en todos los puntos de control (70/30)
CSV_EXTRAIDO = dirs["tablas"] / f"valores_24bandas_{REGION.lower()}.csv"
FORZAR_REEXTRACCION = False

if CSV_EXTRAIDO.exists() and not FORZAR_REEXTRACCION:
    df_all = pd.read_csv(CSV_EXTRAIDO)
    print(f"Tabla cargada desde cache local: {CSV_EXTRAIDO} ({len(df_all)} registros)")
else:
    gpkg_muestras = dirs["muestras"] / f"muestras_limpias_{REGION.lower()}_70_30.gpkg"
    gdf_pts = gpd.read_file(gpkg_muestras)
    fc_pts = geemap.gdf_to_ee(gdf_pts[["sample_id", "clase_id", "clase", "split", "lon", "lat", "geometry"]])

    stack_img = ee.Image(stack_ids["stack"]).select(BANDS["todas"])
    sampled = stack_img.sampleRegions(
        collection=fc_pts,
        properties=["sample_id", "clase_id", "clase", "split", "lon", "lat"],
        scale=int(cfg["stack"]["escala_m"]),
        tileScale=4,
        geometries=False,
    )
    feats = sampled.getInfo()["features"]
    df_all = pd.DataFrame([f["properties"] for f in feats])
    n_bruto = len(df_all)
    df_all = df_all.dropna(subset=BANDS["todas"]).reset_index(drop=True)
    df_all.to_csv(CSV_EXTRAIDO, index=False)
    print(f"Extraidos {n_bruto} puntos -> {len(df_all)} validos (sin nulos). Guardado en {CSV_EXTRAIDO}")

display(pd.crosstab(df_all["clase"], df_all["split"], margins=True))

In [ ]:
# 3. Funcion reutilizable de PCA (Z-Score, Kaiser, Covarianza, Correlacion, Loadings, Contribucion)
def ejecutar_pca_completo(df_subset: pd.DataFrame, columnas: list[str], kaiser_umbral: float = 1.0):
    X = df_subset[columnas].values
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)

    cov_df = pd.DataFrame(np.cov(X_std, rowvar=False), index=columnas, columns=columnas)
    corr_df = df_subset[columnas].corr(method="pearson")

    pca = PCA()
    scores = pca.fit_transform(X_std)
    eigenvalues = pca.explained_variance_
    var_ratio = pca.explained_variance_ratio_
    cum_var = np.cumsum(var_ratio)

    n_kaiser = int(np.sum(eigenvalues >= kaiser_umbral))
    n_kaiser = max(1, n_kaiser)

    pc_names = [f"PC{i+1}" for i in range(len(columnas))]
    eigenvectors_df = pd.DataFrame(pca.components_.T, index=columnas, columns=pc_names)
    # Loadings = eigenvector * sqrt(eigenvalue) (correlacion entre variable y componente)
    loadings_df = eigenvectors_df.multiply(np.sqrt(eigenvalues), axis=1)

    # Contribucion relativa ponderada por varianza en los componentes retenidos (Kaiser)
    # contrib_ik = (eigenvector_ik^2) * var_ratio_k ; normalizada a 100%
    sq_evec = (pca.components_[:n_kaiser, :].T ** 2)
    pesos = var_ratio[:n_kaiser]
    contrib_raw = sq_evec @ pesos
    contrib_pct = 100.0 * contrib_raw / contrib_raw.sum()

    ranking_df = pd.DataFrame(
        {
            "contribucion_pca_pct": contrib_pct,
            "sensor": ["SAR" if (c.startswith("VV") or c.startswith("VH")) else "Optico_S2" for c in columnas],
            "temporada": ["SECA" if c.endswith("_SECA") else "LLUVIA" for c in columnas],
        },
        index=columnas,
    ).sort_values("contribucion_pca_pct", ascending=False)

    eigen_df = pd.DataFrame(
        {
            "Componente": pc_names,
            "Eigenvalor": eigenvalues,
            "Varianza_pct": var_ratio * 100.0,
            "Varianza_acum_pct": cum_var * 100.0,
            "Retenido_Kaiser": eigenvalues >= kaiser_umbral,
        }
    )

    return {
        "n_muestras": len(df_subset),
        "n_kaiser": n_kaiser,
        "cov": cov_df,
        "corr": corr_df,
        "eigen": eigen_df,
        "eigenvectors": eigenvectors_df,
        "loadings": loadings_df,
        "ranking": ranking_df,
        "scores": scores,
    }

# Usamos unicamente puntos de entrenamiento (70%) de clase Mangle (clase_id == 1)
# para evitar fuga de informacion hacia la evaluacion preliminar.
df_mangle_train = df_all[(df_all["split"] == "train") & (df_all["clase_id"] == 1)].copy()
print(f"Muestras de Mangle (train 70%) para PCA: {len(df_mangle_train)}")

kaiser = float(cfg["pca"]["criterio_kaiser"])
pca_bi = ejecutar_pca_completo(df_mangle_train, BANDS["todas"], kaiser_umbral=kaiser)
pca_seca = ejecutar_pca_completo(df_mangle_train, BANDS["seca_todas"], kaiser_umbral=kaiser)
pca_lluvia = ejecutar_pca_completo(df_mangle_train, BANDS["lluvia_todas"], kaiser_umbral=kaiser)

print("\n=== RESUMEN PCA BIESTACIONAL (24 BANDAS) ===")
display(pca_bi["eigen"].head(8))
print(f"Componentes retenidos (Kaiser >= {kaiser}): {pca_bi['n_kaiser']} | Varianza acumulada: {pca_bi['eigen'].loc[pca_bi['n_kaiser']-1, 'Varianza_acum_pct']:.2f}%")

In [ ]:
# 4. Figuras oficiales del Objetivo 1 (300 DPI)
reg_l = REGION.lower()
reg_nom = cfg["regiones"][REGION]["nombre_largo"]

fig1 = plot_correlation_matrix(
    pca_bi["corr"],
    f"Matriz de Correlación de Pearson (24 Bandas S1+S2) — Mangle {reg_nom}",
    save_path=dirs["figuras"] / f"OE1_fig1_correlacion_24b_{reg_l}.png",
)
plt.show()

fig2 = plot_scree_and_variance(
    pca_bi["eigen"]["Eigenvalor"].values,
    pca_bi["eigen"]["Varianza_pct"].values / 100.0,
    f"Análisis de Autovalores y Varianza Explicada (PCA Biestacional) — {reg_nom}",
    kaiser_threshold=kaiser,
    save_path=dirs["figuras"] / f"OE1_fig2_scree_varianza_{reg_l}.png",
)
plt.show()

fig3 = plot_band_contributions(
    pca_bi["ranking"],
    "contribucion_pca_pct",
    f"Contribución Relativa de Bandas S2 y Polarizaciones SAR (PCA Retenido) — {reg_nom}",
    save_path=dirs["figuras"] / f"OE1_fig3_contribucion_bandas_{reg_l}.png",
)
plt.show()

In [ ]:
# 5. Contraste por temporada (12 bandas Seca vs 12 bandas Lluvia) + Separabilidad Multiclase
print("=== COMPARATIVA TEMPORAL DEL PCA ===")
comp_temp = pd.DataFrame([
    {
        "Analisis": "Biestacional (24 bandas)",
        "N_muestras": pca_bi["n_muestras"],
        "PCs_Kaiser": pca_bi["n_kaiser"],
        "Varianza_PC1_pct": pca_bi["eigen"].loc[0, "Varianza_pct"],
        "Varianza_Retenida_pct": pca_bi["eigen"].loc[pca_bi["n_kaiser"] - 1, "Varianza_acum_pct"],
    },
    {
        "Analisis": "Solo Epoca Seca (12 bandas)",
        "N_muestras": pca_seca["n_muestras"],
        "PCs_Kaiser": pca_seca["n_kaiser"],
        "Varianza_PC1_pct": pca_seca["eigen"].loc[0, "Varianza_pct"],
        "Varianza_Retenida_pct": pca_seca["eigen"].loc[pca_seca["n_kaiser"] - 1, "Varianza_acum_pct"],
    },
    {
        "Analisis": "Solo Epoca Lluviosa (12 bandas)",
        "N_muestras": pca_lluvia["n_muestras"],
        "PCs_Kaiser": pca_lluvia["n_kaiser"],
        "Varianza_PC1_pct": pca_lluvia["eigen"].loc[0, "Varianza_pct"],
        "Varianza_Retenida_pct": pca_lluvia["eigen"].loc[pca_lluvia["n_kaiser"] - 1, "Varianza_acum_pct"],
    },
])
display(comp_temp)

# Separabilidad supervisada sobre train (70%): ANOVA F-score (5 clases) y |d de Cohen| Mangle vs Latifoliado
df_train = df_all[df_all["split"] == "train"].copy()
sep_rows = []
for b in BANDS["todas"]:
    grupos = [grp[b].values for _, grp in df_train.groupby("clase_id")]
    f_stat, p_val = stats.f_oneway(*grupos)
    mangle_vals = df_train.loc[df_train["clase_id"] == 1, b].values
    latif_vals = df_train.loc[df_train["clase_id"] == 2, b].values
    if len(latif_vals) > 1 and len(mangle_vals) > 1:
        sp = np.sqrt(((len(mangle_vals)-1)*np.var(mangle_vals, ddof=1) + (len(latif_vals)-1)*np.var(latif_vals, ddof=1)) / (len(mangle_vals)+len(latif_vals)-2))
        d_cohen = abs(np.mean(mangle_vals) - np.mean(latif_vals)) / max(sp, 1e-9)
    else:
        d_cohen = np.nan
    sep_rows.append({
        "banda": b,
        "contribucion_pca_pct": pca_bi["ranking"].loc[b, "contribucion_pca_pct"],
        "ANOVA_F_5clases": f_stat,
        "Cohen_d_Mangle_vs_Latifoliado": d_cohen,
        "sensor": pca_bi["ranking"].loc[b, "sensor"],
        "temporada": pca_bi["ranking"].loc[b, "temporada"],
    })

df_sep = pd.DataFrame(sep_rows).set_index("banda").sort_values("contribucion_pca_pct", ascending=False)
display(df_sep)

In [ ]:
# 6. Seleccion de bandas optimas (OE1) y exportacion de tablas Excel + JSON para Notebook 04
# Criterio explicito y reproducible:
#   Retenemos las bandas cuya contribucion relativa supera la media teorica (100/24 = 4.17%)
#   garantizando representacion de ambos sensores (optico + SAR) y reduciendo redundancia extrema.
umbral_contrib = 100.0 / len(BANDS["todas"])
bandas_pca_sel = df_sep[df_sep["contribucion_pca_pct"] >= umbral_contrib].index.tolist()

# Asegurar que al menos las 2 mejores polarizaciones SAR entren si superan separabilidad
sar_en_sel = [b for b in bandas_pca_sel if b.startswith(("VV", "VH"))]
if len(sar_en_sel) < 2:
    top_sar = df_sep[df_sep["sensor"] == "SAR"].head(2).index.tolist()
    for b in top_sar:
        if b not in bandas_pca_sel:
            bandas_pca_sel.append(b)

print(f"Umbral medio teorico (100/{len(BANDS['todas'])}): {umbral_contrib:.2f}%")
print(f"Bandas seleccionadas por PCA ({len(bandas_pca_sel)}):", bandas_pca_sel)

# Guardar JSON consumido por el Notebook 04
sel_payload = {
    "region": REGION,
    "criterio": f"contribucion_pca_pct >= {umbral_contrib:.2f}% (media teorica) + garantia multisensor",
    "n_componentes_kaiser": pca_bi["n_kaiser"],
    "varianza_retenida_pct": float(pca_bi["eigen"].loc[pca_bi["n_kaiser"] - 1, "Varianza_acum_pct"]),
    "bandas_seleccionadas": bandas_pca_sel,
    "bandas_opticas_sel": [b for b in bandas_pca_sel if not b.startswith(("VV", "VH"))],
    "bandas_sar_sel": [b for b in bandas_pca_sel if b.startswith(("VV", "VH"))],
}
json_path = dirs["tablas"] / f"bandas_seleccionadas_pca_{reg_l}.json"
json_path.write_text(json.dumps(sel_payload, indent=2, ensure_ascii=False), encoding="utf-8")

# Exportar libro Excel completo del Objetivo 1
xlsx_path = dirs["tablas"] / f"OE1_PCA_Resultados_{reg_l}.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    comp_temp.to_excel(writer, sheet_name="Resumen_Temporal", index=False)
    pca_bi["eigen"].to_excel(writer, sheet_name="Eigenvalores_24b", index=False)
    df_sep.to_excel(writer, sheet_name="Ranking_PCA_y_Separab")
    pca_bi["loadings"].to_excel(writer, sheet_name="Loadings_24b")
    pca_bi["eigenvectors"].to_excel(writer, sheet_name="Eigenvectores_24b")
    pca_bi["corr"].to_excel(writer, sheet_name="Correlacion_24b")
    pca_bi["cov"].to_excel(writer, sheet_name="Covarianza_24b")

print(f"[OK] JSON guardado  : {json_path}")
print(f"[OK] Excel guardado : {xlsx_path}")